# White Christmas Analysis — Sault Ste. Marie, Michigan

This notebook explores historical Christmas Day weather data for Sault Ste. Marie, MI (2019–2023), collected via the [Open-Meteo Archive API](https://open-meteo.com/) and stored in a local SQLite database.

**Central question:** Based on past weather patterns, how likely is a white Christmas in Sault Ste. Marie?

For this analysis, a *white Christmas* is defined as a day where:
- measurable precipitation was recorded (> 0 inches), **and**
- the average temperature was at or below freezing (≤ 32°F), making snow more likely than rain.

> **Limitation:** The dataset uses precipitation (rain + snow combined) as a proxy for snowfall. Snowfall-specific data would give a more precise answer.

## 1. Setup and Data Loading

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

conn = sqlite3.connect('weather_data.db')  # connect to the local SQLite database
df = pd.read_sql_query('SELECT * FROM weather_data ORDER BY year', conn)  # load all records into a DataFrame, sorted by year
conn.close()  # close the connection once data is loaded

df['year'] = df['year'].astype(int)  # convert year to integer so it displays cleanly on charts (no decimals)

print(f'Loaded {len(df)} records.')  # confirm how many years of data we have
df.head()  # preview the first few rows

## 2. Data Overview

Each row represents Christmas Day (Dec 25) for one year. The temperature, wind speed, and precipitation columns each hold a 5-year rolling average/min/max up to that year.

In [ ]:
# .describe() gives us count, mean, min, max, and quartiles for each column
df[['avg_temp', 'min_temp', 'max_temp',
    'avg_wind_speed', 'sum_precipitation']].describe().round(2)  # round to 2 decimal places for readability

## 3. Temperature on Christmas Day by Year

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # create the figure and axes to draw on

ax.bar(df['year'], df['avg_temp'], color='steelblue', label='Avg Temp (F)')  # bar chart of average temp per year
ax.errorbar(
    df['year'], df['avg_temp'],
    yerr=[df['avg_temp'] - df['min_temp'], df['max_temp'] - df['avg_temp']],  # error bars show the min/max range around the average
    fmt='none', color='black', capsize=5, label='Min / Max range'
)
ax.axhline(32, color='red', linestyle='--', linewidth=1, label='Freezing (32°F)')  # horizontal line at freezing point for reference

ax.set_title('Christmas Day Temperature — Sault Ste. Marie, MI', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature (°F)')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))  # force x-axis to show whole numbers (years), not decimals
ax.legend()
plt.tight_layout()  # prevent labels from being cut off
plt.show()

## 4. Precipitation on Christmas Day by Year

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bar_colors = ['steelblue' if p > 0 else 'lightgray' for p in df['sum_precipitation']]  # blue if precipitation was recorded, gray if none
ax.bar(df['year'], df['sum_precipitation'], color=bar_colors)

ax.set_title('Christmas Day Precipitation — Sault Ste. Marie, MI', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Precipitation (inches)')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))  # whole number years on x-axis

for _, row in df.iterrows():  # loop through each year to add a label above its bar
    ax.text(row['year'], row['sum_precipitation'] + 0.005,  # position the label just above the top of the bar
            f"{row['sum_precipitation']:.3f}", ha='center', va='bottom', fontsize=9)  # show 3 decimal places

plt.tight_layout()
plt.show()

## 5. White Christmas Probability Calculation

We define a **white christmas** as a year where precipitation was recorded **and** the average temperature was at or below freezing.

In [ ]:
FREEZING = 32.0  # the temperature threshold in Fahrenheit

df['had_precipitation'] = df['sum_precipitation'] > 0  # True if any precipitation was recorded that day
df['was_freezing'] = df['avg_temp'] <= FREEZING  # True if the average temperature was at or below 32 F
df['likely_white_christmas'] = df['had_precipitation'] & df['was_freezing']  # both conditions must be True for a likely white christmas

df[['year', 'avg_temp', 'sum_precipitation', 'had_precipitation', 'was_freezing', 'likely_white_christmas']]  # display the new columns alongside the source data

In [ ]:
total = len(df)  # total number of years analyzed
white_count = df['likely_white_christmas'].sum()  # count how many years met the white christmas criteria
probability = white_count / total  # divide to get the proportion (e.g. 4 out of 5 = 0.80)

print(f'Years analyzed: {total}')
print(f'Likely white christmases: {white_count}')
print(f'Estimated probability: {probability:.0%}')  # format as a percentage

## 6. Summary Chart — White Christmas by Year

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

colors = ['#2ecc71' if wc else '#e74c3c' for wc in df['likely_white_christmas']]  # green for white christmas, red for not
ax.bar(df['year'], [1] * len(df), color=colors, edgecolor='white', linewidth=2)  # all bars same height — this is a status chart, not a quantity chart

for _, row in df.iterrows():  # add a text label centered inside each bar
    label = 'White' if row['likely_white_christmas'] else 'Not white'
    ax.text(row['year'], 0.5, label, ha='center', va='center', color='white', fontweight='bold', fontsize=10)

ax.set_title(f'White Christmas? ({int(df["year"].min())}–{int(df["year"].max())})', fontsize=13)
ax.set_xlabel('Year')
ax.set_yticks([])  # hide the y-axis ticks since bar height has no meaning here
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

print(f'\nBased on {total} years of data, Sault Ste. Marie had a likely white christmas {white_count} out of {total} years ({probability:.0%}).')

## 7. Observations and Next Steps

**What the data shows:**
- All five years recorded measurable precipitation on December 25th.
- All five years had an average temperature at or below freezing.
- Based on this definition, the estimated probability of a white Christmas in Sault Ste. Marie is **100%** across this 5-year window — though that reflects a small sample size.

**Caveats:**
- Five years is a small sample. A longer data range (10–30 years) would produce a more reliable estimate.
- Precipitation includes rain and snow. A day with sleet at 31°F would count here but isn't really a white Christmas.
- Average temperature masking intraday variation — a day could average 30°F but peak above freezing.

**Potential next steps:**
- Extend the dataset to more years (Open-Meteo has data back to 1940).
- Add snowfall-specific data (Open-Meteo provides a `snowfall` variable).
- Compare with the companion ML project to see how the model performs against this baseline.